# Mô hình chính `pair_skip_mlp_specfeat` (đồ thị mức yêu cầu) – thực nghiệm chính

**Settings → Accelerator: GPU T4 x2 hoặc P100** (GPU chỉ để mã hóa BGE-M3; huấn luyện chạy CPU), **Internet: On**. Không cần khóa API: kết quả trích xuất bằng LLM đã có sẵn trong `data/cache/req_cache.json`.

Bấm **Save Version → Save & Run All** (ước tính 4–5 giờ, phần lớn cho resume-job-fit). Notebook chạy 3 thiết lập, mỗi thiết lập 5 seed, cùng fold/validation/seed với báo cáo:

1. Bộ đề tài (1.664 cặp), 5 fold chia theo JD.
2. Bộ đề tài, loại trừ cả hai chiều CV và JD (10 ô).
3. resume-job-fit (7.993 cặp), 5 fold chia theo JD.

Ô cuối in NDCG@10 trên tập kiểm tra. Kết quả trên máy (cùng seed 0–4): **0,886 ± 0,010**, **0,907 ± 0,007**, **0,785 ± 0,015**; báo cáo dùng 10 seed cho resume-job-fit (0,785 ± 0,012). Do huấn luyện trên CPU khác máy, số có thể lệch ở chữ số thứ ba. Các bảng so sánh với baseline và kiểm định (`report/b_eval2.py`, `report/b_eval_ext.py`) cần thêm điểm của các baseline nên chạy trên máy; file điểm của mô hình ở tab Output (`req_jd_kaggle*.csv`, `req_double_kaggle*.csv`, `ext_cna_req_kaggle*.csv`) dùng được trực tiếp cho hai script đó.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv; nproc
!git clone -q -b bang-branch https://github.com/hoangphungvan/doantotnghiep.git
%cd doantotnghiep
!git log --oneline -1

In [ ]:
# Chỉ cài phần còn thiếu, KHÔNG cài lại torch (giữ bản CUDA của Kaggle)
!pip install -q torch_geometric python-dotenv openai pymupdf
import torch, sentence_transformers; print(torch.__version__, torch.cuda.is_available(), sentence_transformers.__version__)

In [ ]:
# resume-job-fit không nằm trong repo: tải train.csv/test.csv gốc từ HuggingFace
from huggingface_hub import snapshot_download
snapshot_download('cnamuangtoun/resume-job-description-fit', repo_type='dataset', local_dir='data/raw/cnamuangtoun')
!ls -la data/raw/cnamuangtoun

In [ ]:
# 1) Bộ đề tài, 5 fold chia theo JD, 5 seed
!mkdir -p /kaggle/working
!python -X utf8 src/modeling/train_req.py jd --seeds 5 --configs pair_skip_mlp_specfeat --tag _kaggle && cp report/results/req_jd_kaggle*.csv /kaggle/working/

In [ ]:
# 2) Bộ đề tài, loại trừ cả hai chiều (2 nhóm CV x 5 nhóm JD), 5 seed
!python -X utf8 src/modeling/train_req.py double --seeds 5 --configs pair_skip_mlp_specfeat --tag _kaggle && cp report/results/req_double_kaggle*.csv /kaggle/working/

In [ ]:
# 3) resume-job-fit, 5 fold chia theo JD, 5 seed. Dòng 'train ... cặp' phải cộng lại 7993 cặp sau khi bỏ trùng. File được ghi sau mỗi seed
!python -X utf8 src/modeling/train_ext.py --protocol jd --seeds 5 --configs pair_skip_mlp_specfeat --tag _kaggle; cp report/results/ext_cna_req_kaggle*.csv /kaggle/working/

In [ ]:
TAG = EXT_TAG = '_kaggle'
# NDCG@10 của mô hình chính trên tập kiểm tra (trung bình ± độ lệch chuẩn theo seed), cùng cách tính với report/b_eval2.py, b_eval_ext.py
import sys, pandas as pd
sys.path.insert(0, '.')
from src.evaluation.metrics import evaluate_ranking
from src.modeling.train_ext import load
own = pd.read_csv('data/raw/pairs_labeled.csv')[['cv_id', 'jd_id', 'grade']]
d = load()
ext = pd.concat([d['train'], d['test']]).drop_duplicates(['cv_id', 'jd_id'])[['cv_id', 'jd_id', 'grade']]
for name, f, lab in (('Bộ đề tài, chia theo JD', 'req_jd' + TAG, own), ('Bộ đề tài, loại trừ hai chiều', 'req_double' + TAG, own), ('resume-job-fit, chia theo JD', 'ext_cna_req' + EXT_TAG, ext)):
    try:
        L = pd.read_csv(f'report/results/{f}.csv')
    except FileNotFoundError:
        print(name, ': chưa chạy'); continue
    L = L[L.config == 'pair_skip_mlp_specfeat'].merge(lab, on=['cv_id', 'jd_id'])
    elig = {q for q, x in L[L.seed == L.seed.min()].groupby('qid') if len(x) >= 2 and (x.grade >= 2).any()}
    r = pd.Series([evaluate_ranking({q: list(zip(y.grade, y.score)) for q, y in x[x.qid.isin(elig)].groupby('qid')}, ks=[10])['ndcg@10']
                   for _, x in L.groupby('seed')])
    print(f'{name}: NDCG@10 = {r.mean():.3f} ± {r.std():.3f} ({len(r)} seed, {len(elig)} truy vấn)')


In [ ]:
!cp report/results/*kaggle*.csv /kaggle/working/ 2>/dev/null; ls -la /kaggle/working/*.csv